# Coherent MXene/metal interface builder

Builds a metal (111) slab on an MXene (0001) in one orthogonal box, with the smallest coherent supercell
within the strain tolerance, and optionally relaxes it with a MACE foundation model.

MXenes and metals are the rows of `data/MXene_FCC_L12_strain_matrix.xlsx`. The building and relaxation
code is `builder.py` and `relax.py` at the top of the repository (linked into this folder), the same code
as the web app. See the repository README for the geometry rules.

- **Inputs:** the MXene, metal and placement, the supercell search and, with `run_relaxation`, the MACE settings.
- **Outputs:** the POSCARs, the supercell match and strain, and, after a relaxation, the energy, gap and work of adhesion.
  `summary` holds everything (`outputs.json` of the command-line tools).

In [ ]:
DESCRIPTION = "Coherent MXene/metal (111) interface: smallest matching supercell, optional MACE relaxation"

In [ ]:
import json
from pathlib import Path

from sim2l import DB, getValidatedInputs, load_yaml_magic   # simtool and yamlmagic, or stand-ins without them

load_yaml_magic()

In [ ]:
EXTRA_FILES = ["builder.py", "relax.py", "sim2l.py"]

In [ ]:
%%yaml INPUTS

mxene:
    description: MXene, a row of the MXenes sheet
    type: Choice
    options: ["Ti2C", "V2C", "Cr2C", "Zr2C", "Nb2C", "Mo2C", "Hf2C", "Ta2C", "W2C", "Ti3C2", "V3C2", "Nb3C2", "Mo3C2", "Ti4C3", "Nb4C3", "Mo2TiC2", "Mo2VC2", "Mo2NbC2", "Mo2TaC2", "Cr2TiC2", "Cr2VC2", "Cr2NbC2", "Cr2TaC2", "Ti2NbC2", "Ti2TaC2", "V2TiC2", "V2TaC2", "Mo2TiC2O2", "Cr2TiC2O2", "Mo2ScC2O2", "Cr2VC2O2", "Ti2VCNO2", "Ti2NbCNO2", "Ti2MoCNO2", "Ti2TaCNO2", "V2TiCNO2", "Nb2TiCNO2", "Zr2NbCNO2", "Mo2TiCNO2", "Mo2NbCNO2", "Cr2TiCNO2"]
    value: V2C

stacking:
    description: MXene stacking, ABC = AβC (octahedral), ABA = AβA (trigonal prismatic)
    type: Choice
    options: [ABC, ABA]
    value: ABC

mxene_rotation:
    description: MXene rotation about z in degrees, 0 or 180 (180 swaps B and C, AβC becomes AγB)
    type: Integer
    value: 0
    min: 0
    max: 180

vacuum:
    description: Vacuum below the MXene
    type: Number
    units: angstrom
    value: 20.0
    min: 0

metal:
    description: FCC metal or L12 compound, a row of the Metals sheet
    type: Choice
    options: ["Ni", "Co", "Cu", "Rh", "Ir", "Pd", "Pt", "Al", "Au", "Ag", "Pb", "Ni3Al", "Ni3Ga", "Ni3Si", "Ni3Fe", "Ni3Mn", "Ni3Pt", "Ni3Ti", "Co3Ti", "Cu3Au", "Pt3Al", "Pt3Ti", "Fe3Al"]
    value: Au

metal_layers:
    description: Metal (111) atomic planes
    type: Integer
    value: 4
    min: 1
    max: 40

metal_site:
    description: Site of the first metal layer; A is over the bottom MXene layer, auto is the site free of the top two MXene layers
    type: Choice
    options: [auto, A, B, C]
    value: auto

metal_stacking:
    description: Metal stacking upward, ABC (continues an MXene at 0°) or ACB (continues one at 180°)
    type: Choice
    options: [ABC, ACB]
    value: ABC

gap:
    description: Top MXene layer to first metal (111) layer
    type: Number
    units: angstrom
    value: 2.3
    min: 0

shift_x:
    description: In-plane shift of the metal along x, from its site
    type: Number
    units: angstrom
    value: 0.0

shift_y:
    description: In-plane shift of the metal along y, from its site
    type: Number
    units: angstrom
    value: 0.0

tol:
    description: Largest |strain| on the metal along x and y (fraction)
    type: Number
    value: 0.02
    min: 0
    max: 0.2

max_replicas:
    description: Largest number of MXene replicas tried per direction
    type: Integer
    value: 10
    min: 1
    max: 30

run_relaxation:
    description: Relax the interface with MACE (off = build only)
    type: Boolean
    value: false

model:
    description: MACE foundation model
    type: Text
    value: mh-1

head:
    description: Output head of the MH models, omat_pbe = PBE(+U), oc20_usemppbe = PBE slabs, matpes_r2scan = r2SCAN
    type: Choice
    options: [omat_pbe, oc20_usemppbe, matpes_r2scan]
    value: omat_pbe

dispersion:
    description: Add D3(BJ) dispersion (needs torch-dftd)
    type: Boolean
    value: false

fmax:
    description: Force convergence
    type: Number
    units: eV/angstrom
    value: 0.01
    min: 0.0001

max_steps:
    description: FIRE steps
    type: Integer
    value: 500
    min: 1

fixed_layers:
    description: Outermost metal layers held fixed
    type: Integer
    value: 2
    min: 0

relax_cell:
    description: Relax a, b and γ (c stays fixed)
    type: Boolean
    value: true

work_of_adhesion:
    description: Compute the work of adhesion from the rigidly separated MXene and metal
    type: Boolean
    value: false

In [ ]:
%%yaml OUTPUTS

formula:
    description: Interface formula
    type: Text

n_atoms:
    description: Number of atoms
    type: Integer

box:
    description: Box lengths along x, y and z
    type: Array
    units: angstrom

strain_x:
    description: Strain on the metal along x (fraction)
    type: Number

strain_y:
    description: Strain on the metal along y (fraction)
    type: Number

match:
    description: MXene and metal replicas, box length and metal strain along x and y
    type: Dict

POSCAR:
    description: Built interface
    type: Text

POSCAR_mxene:
    description: MXene conventional cell
    type: Text

POSCAR_metal:
    description: Metal conventional cubic cell
    type: Text

POSCAR_relaxed:
    description: Relaxed interface (empty without a relaxation)
    type: Text

energy:
    description: Final energy of the relaxed interface (empty without a relaxation)
    type: Number
    units: eV

gap_relaxed:
    description: MXene-metal gap after relaxation (empty without a relaxation)
    type: Number
    units: angstrom

work_of_adhesion:
    description: Work of adhesion (empty unless asked for)
    type: Number
    units: J/m**2

summary:
    description: Every input and output, as outputs.json of the command-line tools
    type: Dict

In [ ]:
defaultInputs = getValidatedInputs(INPUTS)
if defaultInputs:
    globals().update(defaultInputs)

**Computation is below.**

In [ ]:
import builder
import relax

inputs = dict(mxene=mxene, stacking=stacking, mxene_rotation=mxene_rotation, vacuum=vacuum, metal=metal,
              metal_layers=metal_layers, metal_site=metal_site, metal_stacking=metal_stacking, gap=gap,
              shift=[shift_x, shift_y], tol=tol, max_replicas=max_replicas)
if run_relaxation:
    inputs.update(model=model, head=head, dispersion=dispersion, fmax=fmax, max_steps=max_steps,
                  fixed_layers=fixed_layers, relax_cell=relax_cell, work_of_adhesion=work_of_adhesion)
    log = lambda s: print(f"{s['step']:5d}  {s['energy']:.6f} eV  {s['fmax']:.4f} eV/Å") if s["step"] % 25 == 0 else None
    structure, outputs = relax.relax(inputs, progress=log)
else:
    structure, outputs = builder.build(inputs)

for name, text in outputs["files"].items():
    Path(name).write_text(text)
Path("outputs.json").write_text(json.dumps(outputs, indent=2) + "\n")
print(outputs["interface"]["formula"], outputs["match"])

In [ ]:
files, r = outputs["files"], outputs.get("relaxation", {})
db = DB(OUTPUTS)
db.save("formula", outputs["interface"]["formula"])
db.save("n_atoms", outputs["interface"]["n_atoms"])
db.save("box", outputs["interface"]["box"])
db.save("strain_x", outputs["match"]["x"]["strain"])
db.save("strain_y", outputs["match"]["y"]["strain"])
db.save("match", outputs["match"])
db.save("POSCAR", files["POSCAR"])
db.save("POSCAR_mxene", files[f"POSCAR_{mxene}"])
db.save("POSCAR_metal", files[f"POSCAR_{metal}"])
db.save("POSCAR_relaxed", files.get("POSCAR_relaxed"))
db.save("energy", r.get("energy_final"))
db.save("gap_relaxed", r.get("gap"))
db.save("work_of_adhesion", r.get("work_of_adhesion"))
db.save("summary", {k: v for k, v in outputs.items() if k != "files"})